# Task 1 — Inconsistencies and duplicates
CSEN1095 Data Engineering · W26

Goal: turn `club_signups.csv` into a clean table with one row per student per club. The original file is only read, never edited; every fix is done in code.

In [19]:
import hashlib
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

SRC = 'club_signups.csv'
src_hash_before = hashlib.md5(open(SRC, 'rb').read()).hexdigest()

# dtype=str: read everything as text so nothing is silently converted
raw = pd.read_csv(SRC, dtype=str)
df = raw.copy()
rows_start = len(df)
print('Rows at start:', rows_start)

Rows at start: 39


## Part A — Inconsistencies
### A1. Look first

In [20]:
print('shape:', raw.shape)
print()
print(raw.dtypes)

shape: (39, 9)

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object


In [21]:
for col in ['faculty', 'city', 'club', 'fee_paid']:
    print(raw[col].value_counts(dropna=False))
    print()

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1
Name: count, dtype: int64

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1

In [22]:
print('missing values per column:')
print(raw.isna().sum())
print()
print('names written with extra spaces:', (raw['full_name'] != raw['full_name'].str.split().str.join(' ')).sum())
print('emails not lower-case/trimmed   :', (raw['email'] != raw['email'].str.strip().str.lower()).sum())

missing values per column:
signup_id       0
student_id      0
full_name       0
email           0
faculty         0
city            0
club            0
signed_up_at    0
fee_paid        0
dtype: int64

names written with extra spaces: 4
emails not lower-case/trimmed   : 5


Problems visible so far:
- **faculty**: abbreviations in different cases (`iet`, `met`, `ems`, `pharmacy`) plus long names (`Information Engineering and Technology`, ...) and `Pharma`.
- **city**: different cases plus `Alex` and `El Giza`.
- **club**: different cases plus `Chess Club`, `Debating`, `debate club`, `Soccer`.
- **fee_paid**: text in many spellings (`yes`, `Y`, `TRUE`, `1`, `no`, `N`, `false`, `0`).
- **names/emails**: extra spaces, UPPER/lower case.
- **signed_up_at**: still text, probably two formats.

### A2. Faculty, club, city
First the mechanical step (strip, lower-case, collapse repeated spaces), then an explicit dictionary for what is left.

In [23]:
def mechanical(s):
    # strip, lower-case, and collapse any run of whitespace into one space
    return s.str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

for col in ['faculty', 'club', 'city']:
    df[col] = mechanical(df[col])

print('Distinct values after the mechanical step:')
for col in ['faculty', 'club', 'city']:
    print(col, '->', sorted(df[col].unique()))

Distinct values after the mechanical step:
faculty -> ['ems', 'engineering and materials science', 'iet', 'information engineering and technology', 'management', 'media engineering and technology', 'met', 'pharma', 'pharmacy']
club -> ['chess', 'chess club', 'debate', 'debate club', 'debating', 'football', 'music', 'robotics', 'soccer']
city -> ['alex', 'alexandria', 'cairo', 'el giza', 'giza']


In [24]:
faculty_map = {
    'met': 'MET', 'media engineering and technology': 'MET',
    'iet': 'IET', 'information engineering and technology': 'IET',
    'ems': 'EMS', 'engineering and materials science': 'EMS',
    'pharmacy': 'Pharmacy', 'pharma': 'Pharmacy',
    'management': 'Management',
}
club_map = {
    'robotics': 'Robotics',
    'debate': 'Debate', 'debating': 'Debate', 'debate club': 'Debate',
    'music': 'Music',
    'football': 'Football', 'soccer': 'Football',      # Soccer club = Football club
    'chess': 'Chess', 'chess club': 'Chess',
}
city_map = {
    'cairo': 'Cairo',
    'giza': 'Giza', 'el giza': 'Giza',
    'alexandria': 'Alexandria', 'alex': 'Alexandria',
}

df['faculty'] = df['faculty'].map(faculty_map)
df['club']    = df['club'].map(club_map)
df['city']    = df['city'].map(city_map)

# a value missing from a dictionary would become NaN, so check that first
assert df[['faculty', 'club', 'city']].notna().all().all(), 'unmapped value found'

assert set(df['faculty']) <= {'MET', 'IET', 'EMS', 'Pharmacy', 'Management'}
assert set(df['club'])    <= {'Robotics', 'Debate', 'Music', 'Football', 'Chess'}
assert set(df['city'])    <= {'Cairo', 'Giza', 'Alexandria'}

for col in ['faculty', 'club', 'city']:
    print(df[col].value_counts()); print()

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64



### A3. Names and emails
Names: collapse extra spaces, then one capitalisation (Title Case) for everyone. Emails: trimmed and lower-case.

In [25]:
df['full_name'] = (df['full_name'].str.split().str.join(' ')   # trims and collapses spaces
                   .str.title())
df['email'] = df['email'].str.strip().str.lower()
df['student_id'] = df['student_id'].str.strip()

assert not df['full_name'].str.contains(r'\s{2,}|^\s|\s$', regex=True).any()
assert (df['full_name'] == df['full_name'].str.title()).all()
assert (df['email'] == df['email'].str.strip().str.lower()).all()
print(sorted(df['full_name'].unique()))

['Ahmed Adel', 'Ahmed Zaki', 'Ali Ibrahim', 'Farida Ibrahim', 'Hana Zaki', 'Karim Kamel', 'Laila Mostafa', 'Mariam Kamel', 'Mohamed Adel', 'Omar Adel', 'Salma Ibrahim', 'Sara Zaki', 'Youssef Fathy', 'Ziad Saleh']


### A4. fee_paid → boolean

In [26]:
true_values  = {'yes', 'y', 'true', '1'}
false_values = {'no', 'n', 'false', '0'}

fee = df['fee_paid'].str.strip().str.lower()
assert fee.isin(true_values | false_values).all(), 'unknown fee_paid value'
df['fee_paid'] = fee.isin(true_values).astype(bool)

print(df['fee_paid'].dtype)
print(df['fee_paid'].value_counts())

bool
fee_paid
True     27
False    12
Name: count, dtype: int64


### A5. signed_up_at
Two formats are mixed. Find them first.

In [27]:
iso   = df['signed_up_at'].str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$')
slash = df['signed_up_at'].str.match(r'^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$')
print('ISO format   (YYYY-MM-DD HH:MM):', iso.sum())
print('Slash format (??/??/YYYY HH:MM):', slash.sum())
assert (iso | slash).all(), 'a third format exists'

parts = df.loc[slash, 'signed_up_at'].str.extract(r'^(\d{2})/(\d{2})/')
parts = parts.astype(int)
print()
print('slash format, 1st number: min', parts[0].min(), 'max', parts[0].max())
print('slash format, 2nd number: min', parts[1].min(), 'max', parts[1].max())

ISO format   (YYYY-MM-DD HH:MM): 25
Slash format (??/??/YYYY HH:MM): 14

slash format, 1st number: min 13 max 18
slash format, 2nd number: min 9 max 9


The first number goes up to 18, which cannot be a month, so it is the **day**. The second number is always 09 (September), which matches the ISO dates (all `2026-09-13` to `2026-09-18`). So the slash format is **DD/MM/YYYY**. Reading it as MM/DD would be impossible for 13–18 and would silently swap day and month for the rest.

In [28]:
parsed = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
parsed[iso]   = pd.to_datetime(df.loc[iso,   'signed_up_at'], format='%Y-%m-%d %H:%M')
parsed[slash] = pd.to_datetime(df.loc[slash, 'signed_up_at'], format='%d/%m/%Y %H:%M')
df['signed_up_at'] = parsed

assert df['signed_up_at'].notna().all()
print(df['signed_up_at'].dtype)
print('range:', df['signed_up_at'].min(), '->', df['signed_up_at'].max())
assert df['signed_up_at'].dt.month.eq(9).all()   # sanity: everything is in September 2026

datetime64[ns]
range: 2026-09-13 14:36:00 -> 2026-09-18 23:36:00


## Part B — Duplicates
### B1. Exact duplicates

In [29]:
exact_mask = df.duplicated(keep='first')
n_exact = int(exact_mask.sum())
print('Exact duplicate rows:', n_exact)
print(df[df.duplicated(keep=False)].sort_values('signup_id'))

# same count on the untouched raw data? (they are identical text, so yes)
print('Exact duplicates in the raw file:', int(raw.duplicated().sum()))

df = df.drop_duplicates(keep='first')
rows_after_exact = len(df)
print('Rows after removing exact duplicates:', rows_after_exact)

Exact duplicate rows: 3
   signup_id student_id   full_name                          email   faculty   city      club        signed_up_at  fee_paid
0       S015    61-8942   Hana Zaki   hana.zaki@student.guc.edu.eg  Pharmacy   Giza  Football 2026-09-15 13:39:00     False
34      S015    61-8942   Hana Zaki   hana.zaki@student.guc.edu.eg  Pharmacy   Giza  Football 2026-09-15 13:39:00     False
15      S022    61-8224   Sara Zaki   sara.zaki@student.guc.edu.eg       EMS   Giza     Music 2026-09-16 01:39:00      True
24      S022    61-8224   Sara Zaki   sara.zaki@student.guc.edu.eg       EMS   Giza     Music 2026-09-16 01:39:00      True
20      S023    58-1125  Ziad Saleh  ziad.saleh@student.guc.edu.eg       IET  Cairo     Chess 2026-09-16 09:29:00     False
33      S023    58-1125  Ziad Saleh  ziad.saleh@student.guc.edu.eg       IET  Cairo     Chess 2026-09-16 09:29:00     False
Exact duplicates in the raw file: 3
Rows after removing exact duplicates: 36


### B2. The same sign-up submitted again
Keep one row per `student_id` + `club`: the **latest** submission, judged by the parsed timestamp (not by `signup_id`, and not by the raw text, which mixes two formats).

In [30]:
key = ['student_id', 'club']
dups = df[df.duplicated(key, keep=False)].sort_values(key + ['signed_up_at'])
print('Rows involved in a repeated student+club:', len(dups))
dups[['signup_id', 'student_id', 'full_name', 'club', 'signed_up_at', 'fee_paid']]

Rows involved in a repeated student+club: 8


,signup_id,student_id,full_name,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,Football,2026-09-18 23:36:00,True


In [31]:
df = df.sort_values(['signed_up_at', 'signup_id'])
df = df.drop_duplicates(subset=key, keep='last')
rows_after_repeat = len(df)
n_repeat = rows_after_exact - rows_after_repeat
print('Rows removed (same student + club):', n_repeat)
print('Rows left:', rows_after_repeat)

Rows removed (same student + club): 4
Rows left: 32


### B3. Check

In [32]:
assert not df.duplicated(['student_id', 'club']).any()
assert df['signup_id'].is_unique
print('student_id + club is unique ✔')

student_id + club is unique ✔


### Extra checks

In [33]:
# Same name, different students: they must NOT be merged
names = df.groupby('full_name')['student_id'].nunique()
print('Names shared by more than one student_id:')
print(names[names > 1])
print()
print(df[df['full_name'] == 'Mohamed Adel'][['signup_id', 'student_id', 'full_name', 'email', 'faculty', 'city', 'club']]
      .sort_values('student_id'))

# one student_id should describe one person (same name, faculty, city everywhere)
per_student = df.groupby('student_id')[['full_name', 'faculty', 'city']].nunique()
assert (per_student == 1).all().all()
print()
print('Every student_id has exactly one name, faculty and city ✔')

Names shared by more than one student_id:
full_name
Mohamed Adel    2
Name: student_id, dtype: int64

   signup_id student_id     full_name                            email faculty        city      club
14      S002    55-2992  Mohamed Adel  mohamed.adel@student.guc.edu.eg     IET  Alexandria    Debate
18      S024    55-2992  Mohamed Adel  mohamed.adel@student.guc.edu.eg     IET  Alexandria  Robotics
2       S008    61-4844  Mohamed Adel  mohamed.adel@student.guc.edu.eg     EMS       Cairo     Music
23      S009    61-4844  Mohamed Adel  mohamed.adel@student.guc.edu.eg     EMS       Cairo  Robotics
3       S035    61-4844  Mohamed Adel  mohamed.adel@student.guc.edu.eg     EMS       Cairo    Debate

Every student_id has exactly one name, faculty and city ✔


### Order of steps: what if duplicates were removed before fixing the spellings?

In [34]:
# Re-do the duplicate steps on the RAW text (only exact duplicates removed first, no cleaning)
r = raw.drop_duplicates(keep='first')
r_dups = r.duplicated(['student_id', 'club'], keep='last').sum()
print('Raw data: rows', len(raw), '-> after exact duplicates', len(r))
print('Repeats found by student_id + raw club text   :', int(r_dups))
print('Repeats found by student_id + cleaned club    :', n_repeat)
print('Repeats MISSED when cleaning order is wrong   :', n_repeat - int(r_dups))
print()
print('Distinct raw club spellings:', raw['club'].nunique(), ' -> cleaned:', df['club'].nunique())

Raw data: rows 39 -> after exact duplicates 36
Repeats found by student_id + raw club text   : 2
Repeats found by student_id + cleaned club    : 4
Repeats MISSED when cleaning order is wrong   : 2

Distinct raw club spellings: 13  -> cleaned: 5


In [35]:
# Which repeats would be missed? (same student, same club after cleaning, but written differently)
tmp = raw.drop_duplicates(keep='first').copy()
tmp['club_clean'] = tmp['club'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True).map(club_map)
m = tmp[tmp.duplicated(['student_id', 'club_clean'], keep=False)].sort_values(['student_id', 'club_clean'])
m[['signup_id', 'student_id', 'club', 'club_clean', 'signed_up_at', 'fee_paid']]

,signup_id,student_id,club,club_clean,signed_up_at,fee_paid
8,S020,55-9245,Music,Music,2026-09-15 23:07,false
9,S033,55-9245,music,Music,2026-09-16 09:07,1
3,S035,61-4844,Debating,Debate,2026-09-17 11:15,yes
22,S021,61-4844,Debating,Debate,16/09/2026 01:15,no
25,S034,61-7344,debate club,Debate,16/09/2026 02:01,TRUE
35,S017,61-7344,Debate,Debate,15/09/2026 18:01,false
21,S032,61-7344,football,Football,2026-09-17 23:36,false
38,S036,61-7344,football,Football,18/09/2026 23:36,1


## Save the result

In [36]:
clean = (df.sort_values('signup_id')
           .reset_index(drop=True)
           [['signup_id', 'student_id', 'full_name', 'email', 'faculty', 'city', 'club', 'signed_up_at', 'fee_paid']])

clean.to_csv('club_signups_clean.csv', index=False, date_format='%Y-%m-%d %H:%M')

# the source file must be unchanged
assert hashlib.md5(open(SRC, 'rb').read()).hexdigest() == src_hash_before

check = pd.read_csv('club_signups_clean.csv', parse_dates=['signed_up_at'])
print(check.shape, dict(check.dtypes.astype(str)))
print()
print('Started with      :', rows_start)
print('Exact duplicates  :', n_exact, '->', rows_after_exact)
print('Repeated sign-ups :', n_repeat, '->', rows_after_repeat)
print('Final rows        :', len(clean))
clean

(32, 9) {'signup_id': 'object', 'student_id': 'object', 'full_name': 'object', 'email': 'object', 'faculty': 'object', 'city': 'object', 'club': 'object', 'signed_up_at': 'datetime64[ns]', 'fee_paid': 'bool'}

Started with      : 39
Exact duplicates  : 3 -> 36
Repeated sign-ups : 4 -> 32
Final rows        : 32


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False
